In [2]:
# ============================================================
# SIG-05 AUTOMATRIX
# Mini Integration Project – Smart Light Classifier
# ============================================================

import tensorflow as tf
import numpy as np
import os

print("==============================================")
print("   SMART LIGHT CLASSIFIER - TinyML")
print("==============================================")

print("TensorFlow Version:", tf.__version__)

# ============================================================
# 1. CREATE LDR TRAINING DATA
# ============================================================
# ESP32 LDR ADC range: 0 - 4095
#
# 0 - 1200       = DARK
# 1201 - 2800    = NORMAL
# 2801 - 4095    = BRIGHT

np.random.seed(42)

dark = np.random.randint(100, 1200, 300)
normal = np.random.randint(1201, 2800, 300)
bright = np.random.randint(2801, 4095, 300)

X = np.concatenate([
    dark,
    normal,
    bright
]).astype(np.float32)

y = np.array(
    [0] * 300 +
    [1] * 300 +
    [2] * 300
)

# Normalize LDR values
X = X / 4095.0

# Reshape
X = X.reshape(-1, 1)

print("\nTraining samples:", len(X))

# ============================================================
# 2. CREATE TINYML MODEL
# ============================================================

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# 3. TRAIN MODEL
# ============================================================

print("\nTraining model...")

model.fit(
    X,
    y,
    epochs=50,
    batch_size=16,
    verbose=0
)

print("Model training completed!")

# ============================================================
# 4. CHECK ACCURACY
# ============================================================

loss, accuracy = model.evaluate(
    X,
    y,
    verbose=0
)

print(
    "Training Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

# ============================================================
# 5. TEST SAMPLE LDR VALUES
# ============================================================

classes = [
    "DARK",
    "NORMAL",
    "BRIGHT"
]

test_values = [
    500,
    2000,
    3500
]

print("\n==============================================")
print("          SAMPLE PREDICTIONS")
print("==============================================")

for value in test_values:

    sample = np.array(
        [[value / 4095.0]],
        dtype=np.float32
    )

    prediction = model.predict(
        sample,
        verbose=0
    )

    predicted_class = np.argmax(
        prediction[0]
    )

    confidence = prediction[0][predicted_class]

    # Automatic LED control logic
    if predicted_class == 0:
        led_status = "ON"
    elif predicted_class == 1:
        led_status = "CONTROLLED"
    else:
        led_status = "OFF"

    print(
        "LDR:",
        value,
        "| Class:",
        classes[predicted_class],
        "| LED:",
        led_status,
        "| Confidence:",
        round(float(confidence), 3)
    )

# ============================================================
# 6. CONVERT TO TENSORFLOW LITE
# ============================================================

print("\nConverting model to TFLite...")

converter = tf.lite.TFLiteConverter.from_keras_model(
    model
)

tflite_model = converter.convert()

# Save TFLite model
with open(
    "light_model.tflite",
    "wb"
) as f:
    f.write(tflite_model)

print("light_model.tflite created successfully!")

# ============================================================
# 7. CONVERT TFLITE TO C HEADER
# ============================================================

os.system(
    "xxd -i light_model.tflite > light_model_data.h"
)

print("light_model_data.h created successfully!")

# ============================================================
# 8. CHECK GENERATED FILES
# ============================================================

print("\n==============================================")
print("             GENERATED FILES")
print("==============================================")

print(
    "light_model.tflite :",
    os.path.getsize("light_model.tflite"),
    "bytes"
)

print(
    "light_model_data.h :",
    os.path.getsize("light_model_data.h"),
    "bytes"
)

print("\n==============================================")
print("        SIG-05 COMPLETED SUCCESSFULLY")
print("==============================================")

   SMART LIGHT CLASSIFIER - TinyML
TensorFlow Version: 2.21.0

Training samples: 900

Training model...
Model training completed!
Training Accuracy: 97.78 %

          SAMPLE PREDICTIONS
LDR: 500 | Class: DARK | LED: ON | Confidence: 0.988
LDR: 2000 | Class: NORMAL | LED: CONTROLLED | Confidence: 0.986
LDR: 3500 | Class: BRIGHT | LED: OFF | Confidence: 0.995

Converting model to TFLite...
Saved artifact at '/tmp/tmp2n1ssn53'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  134785541493520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134785541504656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134785541504848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134785541494096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134785541504464: TensorSpec(shape=(), dtype=tf.res